In [0]:
"""
05_packaging_events.py

Creates the Silver Packaging Events table.

Input:
    parsed_events

Output:
    packaging_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Packaging Events
# ============================================================

@dp.table(
    name="packaging_events",
    comment="Validated manufacturing packaging events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_execution_id",
    "execution_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_serial_number",
    "serial_number IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_package_id",
    "package_id IS NOT NULL",
)

@dp.expect(
    "positive_package_weight",
    "package_weight_kg > 0",
)

@dp.expect(
    "valid_packaging_status",
    "packaging_status IN ('READY_FOR_SHIPMENT', 'PACKAGED', 'HOLD')",
)

def packaging_events():

    df = spark.readStream.table("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only packaging events
        # -----------------------------------------

        .filter(
            col("event_type") == "PACKAGING_COMPLETED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "execution_id",
            "serial_number",

            "product_code",

            "source_system",
            "correlation_id",

            "silver_processing_timestamp",

            "payload.package_id",

            "payload.package_type",

            "payload.package_weight_kg",

            "payload.package_length_mm",
            "payload.package_width_mm",
            "payload.package_height_mm",

            "payload.packaging_status",

            "payload.product_name",
            "payload.family",

        )

    )